In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

df_list = []
for w in range(4):
    df = pd.read_csv(f'../data/hospitalizations/hospitalizations_{w+1}wk.csv', index_col=0)
    df = df[df.geo_value == 'us']
    df.drop(columns=['geo_value'], inplace=True)
    print(f"{w+1}wk data shape: {df.shape}")
    print(f"Target date range: {df.target_end_date.min()} to {df.target_end_date.max()}")
    valid_dates = sorted(df.target_end_date[df.actual.notna()].unique())
    print(f"First 5 dates with non-NA Y: {valid_dates[:5]}")
    print(f"Last 5 dates with non-NA Y: {valid_dates[-5:]}")
    print(f"Range of ahead: {df.ahead.min()} to {df.ahead.max()}")
    df_list.append(df)

1wk data shape: (19075, 30)
Target date range: 2020-03-28 to 2023-06-12
First 5 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29', '2020-07-30', '2020-07-31']
Last 5 dates with non-NA Y: ['2023-06-06', '2023-06-07', '2023-06-08', '2023-06-09', '2023-06-10']
Range of ahead: 1 to 7
2wk data shape: (19048, 30)
Target date range: 2020-04-04 to 2023-06-19
First 5 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29', '2020-07-30', '2020-07-31']
Last 5 dates with non-NA Y: ['2023-06-06', '2023-06-07', '2023-06-08', '2023-06-09', '2023-06-10']
Range of ahead: 8 to 14
3wk data shape: (18433, 30)
Target date range: 2020-04-11 to 2023-06-26
First 5 dates with non-NA Y: ['2020-07-27', '2020-07-28', '2020-07-29', '2020-07-30', '2020-07-31']
Last 5 dates with non-NA Y: ['2023-06-06', '2023-06-07', '2023-06-08', '2023-06-09', '2023-06-10']
Range of ahead: 15 to 21
4wk data shape: (18139, 30)
Target date range: 2020-04-18 to 2023-07-03
First 5 dates with non-NA Y: ['2020-07-27',

In [46]:
df_list2 = []
forecasts_to_take = set()

for w, df in enumerate(df_list):
    df2 = df[(df.target_end_date >= '2020-07-27') & (df.target_end_date <= '2023-06-10')]
    print(f"{w+1}wk data")
    print("# of rows after filtering dates:", df2.shape)   # Actual column has no NA after this
    forecasters, counts = np.unique(df2.forecaster, return_counts=True)
    sorted_indices = np.argsort(counts)[::-1]
    sorted_forecasters = forecasters[sorted_indices]
    sorted_counts = counts[sorted_indices]
    print(sorted_forecasters[0:10])
    print(sorted_counts[0:10])

    df_list2.append(df2)

    if len(forecasts_to_take) == 0:
        forecasts_to_take.update(sorted_forecasters[0:10])
    else:
        forecasts_to_take.intersection_update(sorted_forecasters[0:10])

print("-" * 100)

forecaster_list = list(forecasts_to_take)
print("Forecasters that has plenty of data for all wk predictions:")
print(forecaster_list)

print("-" * 100)

for idx, df in enumerate(df_list2):
    df_list2[idx] = df[df.forecaster.isin(forecaster_list)]
    print(f"{idx+1}wk data")
    print("# of rows after filtering forecasters:", df_list2[idx].shape)

1wk data
# of rows after filtering dates: (18384, 30)
['CU-select' 'GT-DeepCOVID' 'COVIDhub_CDC-ensemble'
 'COVIDhub-4_week_ensemble' 'COVIDhub-baseline' 'COVIDhub-ensemble'
 'Karlen-pypm' 'JHU_IDD-CovidSP' 'MOBS-GLEAM_COVID'
 'COVIDhub-trained_ensemble']
[1043  979  915  915  915  915  875  825  803  796]
2wk data
# of rows after filtering dates: (18307, 30)
['CU-select' 'GT-DeepCOVID' 'COVIDhub_CDC-ensemble'
 'COVIDhub-4_week_ensemble' 'COVIDhub-baseline' 'COVIDhub-ensemble'
 'Karlen-pypm' 'JHU_IDD-CovidSP' 'MOBS-GLEAM_COVID'
 'COVIDhub-trained_ensemble']
[1050  979  908  908  908  908  875  826  796  789]
3wk data
# of rows after filtering dates: (17645, 30)
['CU-select' 'GT-DeepCOVID' 'COVIDhub-4_week_ensemble' 'COVIDhub-baseline'
 'COVIDhub_CDC-ensemble' 'Karlen-pypm' 'JHU_IDD-CovidSP'
 'MOBS-GLEAM_COVID' 'COVIDhub-trained_ensemble' 'USC-SI_kJalpha']
[1057  981  901  901  901  875  832  789  782  770]
4wk data
# of rows after filtering dates: (17309, 30)
['CU-select' 'GT-DeepCOVID

In [47]:
# Imputation
g = df_list2[0].groupby("target_end_date")["actual"]

result = pd.DataFrame({
    "actual": g.first(),                       # candidate value
    "any_na": g.apply(lambda x: x.isna().any()),
    "n_unique_non_na": g.nunique(dropna=True)  # number of unique actual values
})

result["all_equal"] = result["n_unique_non_na"] == 1
result["valid"] = (~result["any_na"]) & (result["all_equal"])
print(f'Start date: {result.index.min()}, End date: {result.index.max()}, # of dates in data: {result.shape[0]}')
print(f'Total dates: {(pd.to_datetime(result.index.max()) - pd.to_datetime(result.index.min())).days + 1}')
print(f'Not valid dates: {result.shape[0] - result["valid"].sum()}')


alpha_list = [float(col[10:]) for col in df_list2[0].columns if col.startswith('forecast_0')]
Y = result['actual'].sort_index()
Y_impute = {}
for w in range(1,5):
    Y_impute[w] = {}
    Y_delayed = Y.shift(7*w).fillna(0)
    Y_7days_std_delayed = Y.rolling(7).std().shift(7*w).fillna(0) 
    for alpha in alpha_list:
        Y_impute[w][alpha] = Y_delayed + Y_7days_std_delayed * stats.norm.ppf(alpha)

Start date: 2020-07-27, End date: 2023-06-10, # of dates in data: 1049
Total dates: 1049
Not valid dates: 0


In [60]:
forecaster_list = ['JHU_IDD-CovidSP', 'CU-select', 'COVIDhub-baseline', 'Karlen-pypm', 'MOBS-GLEAM_COVID', 'GT-DeepCOVID'] + ['COVIDhub-4_week_ensemble']

# Given df, take the latest forecast
def take_last_among_dup (df):
    assert all(col in df.columns for col in ['target_end_date', 'forecast_date'])
    return df.loc[df.groupby('target_end_date')['forecast_date'].idxmax()]

df_data_dic = {}

for w in range(4):
    print("-" * 100)
    print(f"Processing {w+1}wk data")
    df_data_dic[w+1] = {}

    print('Target date with non-zero std Actual (across different forecasters): ', np.sum(df_list2[w].groupby('target_end_date').actual.std() > 0))

    for f_name in forecaster_list:
        df_here = df_list2[w][df_list2[w].forecaster == f_name]   
        date_start = pd.to_datetime(df_here.target_end_date.min())
        date_end = pd.to_datetime(df_here.target_end_date.max())
        date_range_length = (date_end - date_start).days + 1  # inclusive of endpoints
        
        # Duplicate check 
        ud, rc = np.unique(df_here.target_end_date, return_counts=True)
        n_missing = date_range_length - len(ud)
        print(f'{f_name:>25}: Rows {df_here.shape[0]:>5}, First date: {date_start.date()}, Last date: {date_end.date()}, Missing: {n_missing}')
        missing_dates = pd.date_range(start=date_start, end=date_end).strftime('%Y-%m-%d').difference(df_here.target_end_date)
        if False:
            print(missing_dates)
        if np.sum(rc > 1) >= 1:
            dup_dates = np.sort(ud[rc > 1])
            print(f"{0:>27}Contains {len(dup_dates)} duplicate forecasts from {dup_dates[0]} to {dup_dates[-1]}")
            df_here = take_last_among_dup(df_here)
        
        df_data_dic[w+1][f_name] = df_here.drop(columns=['ahead', 'forecaster', 'forecast_date', 'forecast_NaN', 'actual'])

----------------------------------------------------------------------------------------------------
Processing 1wk data
Target date with non-zero std Actual (across different forecasters):  0
          JHU_IDD-CovidSP: Rows   825, First date: 2020-07-27, Last date: 2023-06-10, Missing: 226
                          0Contains 2 duplicate forecasts from 2021-08-23 to 2022-08-22
                CU-select: Rows  1043, First date: 2020-07-27, Last date: 2023-03-12, Missing: 28
                          0Contains 112 duplicate forecasts from 2020-08-07 to 2020-12-31
        COVIDhub-baseline: Rows   915, First date: 2020-12-08, Last date: 2023-06-10, Missing: 0
              Karlen-pypm: Rows   875, First date: 2020-07-27, Last date: 2023-02-26, Missing: 70
         MOBS-GLEAM_COVID: Rows   803, First date: 2020-12-29, Last date: 2023-06-10, Missing: 92
                          0Contains 1 duplicate forecasts from 2021-05-03 to 2021-05-03
             GT-DeepCOVID: Rows   979, First date: 

In [ ]:

# covidhub_date_range = pd.date_range(start=df_covidhub_4wk.target_end_date.min(), end=df_covidhub_4wk.target_end_date.max())

# df3 = pd.concat([df_data_dic[f_name] for f_name in forecaster_list])
# is_data_missing = df3.groupby('target_end_date').size() != len(f_list)
# print('Days with missing forecasts:', np.sum(is_data_missing))    
# print(is_data_missing[~is_data_missing])
# print('Target date with NA actual: ', np.sum(df3.groupby('target_end_date')['actual'].apply(lambda x: x.isna().any())))


# For weekely data
# print(df3.groupby('forecast_date').size())
# print('Days with missing forecasts:', np.sum(df3.groupby('forecast_date').size() != 28))    
# df3 = df3[df3.forecast_date != '2023-06-05']
# print('Days with missing forecasts (after removing 2023-06-05):', np.sum(df3.groupby('forecast_date').size() != 28))

In [ ]:
df_covidhub_4wk = df_data_dic[4]['COVIDhub-baseline']
start_date = df_covidhub_4wk.target_end_date.min()
end_date = df_covidhub_4wk.target_end_date.max()
dates_list = pd.date_range(start=start_date, end=end_date).strftime('%Y-%m-%d')

forecasts_dict = {}
for w in range(1,5):
    forecasts_dict[w] = {}
    for f_name in forecaster_list:
        forecasts_dict[w][f_name] = {}
        for alpha in alpha_list:
            forecasts_dict[w][f_name][alpha] = (df_data_dic[w][f_name][[f'forecast_{alpha}', 'target_end_date']]
                                                    .set_index('target_end_date')[f'forecast_{alpha}']
                                                    .reindex(dates_list)
                                                    .fillna(Y_impute[w][alpha])
                                                    )

# Weekly aggregate
# df4 = df3.drop(columns=['ahead', 'target_end_date', 'forecast_NaN']).groupby(['forecaster', 'forecast_date'], as_index=False).sum()
# Y = {date: df4.loc[df4['forecast_date'] == date, 'actual'].values[0] for date in sorted(df4['forecast_date'].unique())}

In [68]:
import pickle
d = {}
d['forecasts_dict'] = forecasts_dict
d['alpha_list'] = alpha_list
d['dates_list'] = dates_list
d['forecaster_list'] = forecaster_list
d['Y'] = Y

pickle.dump(d, open('../data/hospitalizations/preprocess_data.pkl', 'wb'))